# Chapter 01 복습퀴즈

- 이름: 안형준
- GitHub ID: apll970502
- 작성날짜: 2026-09-10

## 오늘의 목표
DataFrame으로 준비된 고객 데이터를 Python의 기본 반복문과 조건문으로 직접 처리한다.
pandas의 집계 기능을 사용하지 않고 날짜의 최대/최소, 나이의 합계/평균,
컬럼명의 소문자 변환을 직접 구현한다.

## 1. 문제 이해

### 사용 데이터
- 데이터: `customers.csv` → `records` (고객 150명이 들어 있는 List)
- 날짜 컬럼: `Signup_Date`
- 숫자 컬럼: `Age`

### 최종적으로 구해야 하는 결과
1. 가장 오래된 가입일
2. 가장 최근 가입일
3. 전체 나이 합계
4. 전체 나이 평균
5. 소문자로 바꾼 컬럼명 List

### 이번 문제에서 사용하면 안 되는 기능
- 준비 셀 이후 pandas 집계 기능 (`df["Age"].sum()`, `.mean()`, `.max()`, `.min()`, `df.describe()`, `df.columns.str.lower()`)
- Python 내장 `max()`, `min()`, `sum()`
- `statistics.mean()`

### 이 문제를 내 말로 다시 설명하면
고객 한 명 한 명이 Dictionary로 들어 있는 List를 처음부터 끝까지 돌면서,
가입일은 "지금까지 본 것 중 가장 이른 날짜 / 가장 늦은 날짜"를 계속 바꿔 기억하고,
나이는 하나씩 더해 합계를 만들고 사람 수도 같이 센 뒤 마지막에 나눠 평균을 낸다.
컬럼명은 이름 하나씩 `lower()`로 바꿔 새 List에 담는다.
즉 pandas가 한 줄로 해주던 일을 **반복 + 변수 + 조건**으로 직접 만드는 문제다.

## 2. 작업 계획

### 전체 작업 순서
1. 고객 데이터를 준비한다. (`records`, `columns`)
2. 고객을 한 명씩 확인할 수 있는지 먼저 본다.
3. 가장 오래된 가입일, 가장 최근 가입일을 기억할 변수를 첫 번째 고객의 가입일로 준비한다.
4. 고객을 한 명씩 보면서 현재 가입일이 더 이르면 오래된 가입일을, 더 늦으면 최근 가입일을 바꾼다.
5. 나이 합계 변수와 사람 수 변수를 0으로 준비한다.
6. 고객을 한 명씩 보면서 나이를 합계에 더하고 사람 수를 1 늘린다.
7. 반복이 끝난 뒤 합계 ÷ 사람 수로 평균을 계산한다.
8. 컬럼명을 하나씩 소문자로 바꿔 새 List에 추가하고, 모든 결과를 출력한다.

### 반복해야 하는 것은 무엇인가?
- `records` 안의 고객 한 명(Dictionary)
- `columns` 안의 컬럼명 하나(문자열)

### 반복하면서 계속 기억해야 하는 값은 무엇인가?
- 지금까지 가장 오래된 가입일, 가장 최근 가입일
- 지금까지 더한 나이 합계, 지금까지 확인한 사람 수
- 소문자로 바꾼 컬럼명을 모아 둘 List

### 조건으로 비교해야 하는 값은 무엇인가?
- 현재 고객의 가입일 vs 지금까지 가장 오래된 가입일 (`<`)
- 현재 고객의 가입일 vs 지금까지 가장 최근 가입일 (`>`)

### 반복이 모두 끝난 뒤 계산해야 하는 것은 무엇인가?
- 평균 = 나이 합계 ÷ 사람 수

## 3. 수도코드

```text
가장 오래된 가입일 = 첫 번째 고객의 가입일
가장 최근 가입일   = 첫 번째 고객의 가입일
나이 합계 = 0
사람 수   = 0

전체 고객에 대해 반복한다
    - 현재 고객의 가입일과 나이를 꺼낸다
    - 만약 현재 가입일 < 가장 오래된 가입일 이면
          가장 오래된 가입일 = 현재 가입일
    - 만약 현재 가입일 > 가장 최근 가입일 이면
          가장 최근 가입일 = 현재 가입일
    - 나이 합계 = 나이 합계 + 현재 나이
    - 사람 수 = 사람 수 + 1

반복이 끝나면
    나이 평균 = 나이 합계 / 사람 수

소문자 컬럼명 = 빈 List
컬럼명에 대해 반복한다
    - 현재 컬럼명을 소문자로 바꾼다
    - 소문자 컬럼명 List 뒤에 추가한다

최종 결과를 출력한다
```

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/GilbertMoon/llm-data-analysis-course/main/data/raw/customers.csv"

df = pd.read_csv(url)

df.columns = [
    "Customer_ID",
    "Name",
    "Gender",
    "Age",
    "City",
    "Signup_Date"
]

records = df.to_dict(orient="records")
columns = list(df.columns)

print(df.head())

   Customer_ID Name Gender  Age City Signup_Date
0            1  김수민      F   19   광주  2024-06-19
1            2  김정호      F   32   대구  2025-11-02
2            3  이경수      F   61   성남  2024-06-12
3            4  조영호      F   55   울산  2026-04-13
4            5  이예원      F   19   부산  2024-09-13


In [2]:
# 준비된 변수의 모습 관찰 (여기서부터 pandas 사용 X)
print(type(records))      # 전체 고객
print(len(records))       # 한 행 = 고객 한 명
print(type(records[0]))   # 고객 한 명
print(records[0])
print(type(columns))
print(columns)

<class 'list'>
150
<class 'dict'>
{'Customer_ID': 1, 'Name': '김수민', 'Gender': 'F', 'Age': 19, 'City': '광주', 'Signup_Date': '2024-06-19'}
<class 'list'>
['Customer_ID', 'Name', 'Gender', 'Age', 'City', 'Signup_Date']


## 4. 데이터 준비 확인

### `records`는 어떤 자료구조인가?
List. 고객 150명이 순서대로 들어 있다.

### 고객 한 명은 어떤 자료구조인가?
Dictionary. `{'Customer_ID': 1, 'Name': '김수민', ..., 'Signup_Date': '2024-06-19'}`처럼
컬럼명이 Key, 그 고객의 값이 Value다. 그래서 `record["Age"]`처럼 Key로 값을 꺼낸다.

### `columns`는 어떤 자료구조인가?
문자열 6개가 들어 있는 List. `['Customer_ID', 'Name', 'Gender', 'Age', 'City', 'Signup_Date']`

### Chapter 01에서 연습한 어떤 구조와 비슷한가?
`students = [{"name": ..., "score": ..., "attendance": ...}, ...]`처럼
**Dictionary 여러 개가 들어 있는 List** 구조와 같다.
for로 한 명씩 꺼내고 Key로 값을 읽는 방식이 그대로 쓰인다.

# 5. 직접 구현

## 12-1. 전체 고객을 한 명씩 확인하기
계산 없이, 고객이 하나씩 나오는지와 Key로 값을 읽을 수 있는지만 확인한다.
(150명을 다 출력하면 길어서 처음 3명만 출력)

In [3]:
count = 0

for record in records:
    count = count + 1
    if count <= 3:
        print(record)
        print("가입일:", record["Signup_Date"], "/ 나이:", record["Age"])

print("확인한 고객 수:", count)

{'Customer_ID': 1, 'Name': '김수민', 'Gender': 'F', 'Age': 19, 'City': '광주', 'Signup_Date': '2024-06-19'}
가입일: 2024-06-19 / 나이: 19
{'Customer_ID': 2, 'Name': '김정호', 'Gender': 'F', 'Age': 32, 'City': '대구', 'Signup_Date': '2025-11-02'}
가입일: 2025-11-02 / 나이: 32
{'Customer_ID': 3, 'Name': '이경수', 'Gender': 'F', 'Age': 61, 'City': '성남', 'Signup_Date': '2024-06-12'}
가입일: 2024-06-12 / 나이: 61
확인한 고객 수: 150


### 날짜 최대/최소 구현 전 생각
- 반복할 값: `records` 안의 고객 한 명
- 비교할 값: 현재 고객의 `Signup_Date` 문자열
- 반복 전에 준비할 변수: `oldest_date`, `newest_date` — 둘 다 **첫 번째 고객의 가입일**로 시작
  (0이나 빈 문자열로 시작하면 비교 기준이 실제 데이터가 아니게 된다)
- 어떤 경우에 값을 바꿀 것인가:
  - 현재 가입일 `<` `oldest_date` → `oldest_date`를 현재 가입일로 바꾼다
  - 현재 가입일 `>` `newest_date` → `newest_date`를 현재 가입일로 바꾼다
- 아직 헷갈리는 부분: 날짜를 문자열로 비교해도 되는 이유 → `YYYY-MM-DD`는 자릿수가 고정이고
  큰 단위(연 → 월 → 일)가 앞에 있어서, 앞 글자부터 비교하는 문자열 비교 순서가 날짜 순서와 같다.

In [4]:
oldest_date = records[0]["Signup_Date"]
newest_date = records[0]["Signup_Date"]

for record in records:
    signup_date = record["Signup_Date"]

    if signup_date < oldest_date:
        oldest_date = signup_date

    if signup_date > newest_date:
        newest_date = signup_date

print("가장 오래된 가입일:", oldest_date)
print("가장 최근 가입일:", newest_date)

가장 오래된 가입일: 2023-07-14
가장 최근 가입일: 2026-06-27


### 나이 합계/평균 구현 전 생각
- 합계를 저장할 변수: `total_age`
- 처음 값은 무엇으로 둘 것인가: `0` (아직 아무도 더하지 않았으므로)
- 고객 한 명을 확인할 때 무엇을 더할 것인가: 현재 고객의 `Age`
- 사람 수는 어떻게 셀 것인가: `customer_count`를 0으로 시작해 고객 한 명마다 1씩 더한다
- 평균은 언제 계산할 것인가: **반복이 모두 끝난 뒤** 한 번 (`total_age / customer_count`)
  반복 안에서 나누면 중간 평균이 계속 덮어써질 뿐 의미가 없다
- 아직 헷갈리는 부분:

In [5]:
total_age = 0
customer_count = 0

for record in records:
    age = record["Age"]
    total_age = total_age + age
    customer_count = customer_count + 1

# 반복이 끝난 뒤 평균 계산
average_age = total_age / customer_count

print("확인한 고객 수:", customer_count)
print("전체 나이 합계:", total_age)
print("전체 나이 평균:", average_age)

확인한 고객 수: 150
전체 나이 합계: 6313
전체 나이 평균: 42.086666666666666


### 컬럼명 소문자 변환 전 생각
- 반복할 대상: `columns` 안의 컬럼명 문자열
- 새 컬럼명을 저장할 자료구조: 빈 List `lower_columns = []`
- 문자열 하나를 소문자로 바꾸는 방법: `문자열.lower()`
- 바뀐 문자열을 어디에 추가할 것인가: `lower_columns.append(...)`로 List 맨 뒤에 추가
- 아직 헷갈리는 부분: `lower()`는 원래 문자열을 바꾸지 않고 **새 문자열을 돌려준다**
  → 그래서 결과를 받아서 append 해야 한다

In [6]:
lower_columns = []

for column in columns:
    lower_column = column.lower()
    lower_columns.append(lower_column)

print("원래 컬럼명:", columns)
print("소문자 컬럼명:", lower_columns)

원래 컬럼명: ['Customer_ID', 'Name', 'Gender', 'Age', 'City', 'Signup_Date']
소문자 컬럼명: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']


# 6. 실행 결과

In [7]:
print("가장 오래된 가입일:", oldest_date)
print("가장 최근 가입일  :", newest_date)
print("전체 나이 합계    :", total_age)
print(f"전체 나이 평균    : {average_age:.2f}")
print("소문자 컬럼명     :", lower_columns)

가장 오래된 가입일: 2023-07-14
가장 최근 가입일  : 2026-06-27
전체 나이 합계    : 6313
전체 나이 평균    : 42.09
소문자 컬럼명     : ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']


### 결과 확인
- 첫 셀부터 마지막 셀까지 오류 없이 실행되었다.
- 고객 150명 전체를 반복했다. (`customer_count` = 150)
- 가장 오래된 가입일 `2023-07-14`, 가장 최근 가입일 `2026-06-27`
- 나이 합계 `6313`, 평균 `6313 / 150 = 42.09`
- 컬럼명 6개가 모두 소문자로 바뀌었다.
- 준비 셀 이후 pandas 집계 기능, `max()`, `min()`, `sum()`, `mean()`을 사용하지 않았다.

## 추가 학습 기록 1

### 무엇이 필요했는가?

### 처음에는 무엇을 몰랐는가?

### 내가 먼저 생각한 내용

### 어디에서 확인했는가?
- Python 공식 문서 / 수업 자료 / AI / 기타

### 확인 후 내가 이해한 내용

### 이 내용을 실제 코드의 어느 부분에 사용했는가?

## 아직 이해가 안 되는 부분

### 항목 1
- 이해되지 않는 내용:
- 어디까지는 이해했는가:
- 정확히 어느 부분부터 모르겠는가:

### 항목 2
- 이해되지 않는 내용:
- 어디까지는 이해했는가:
- 정확히 어느 부분부터 모르겠는가:

# AI 사용 기록

## AI 질문 1

### 질문하기 전에 내가 생각한 내용

### AI에게 실제로 한 질문

### AI의 답변에서 이해한 핵심 내용

### 이 답변이 내 문제 해결에 어떤 도움을 주었는가?

### AI가 정답 또는 완성 코드를 작성했는가?


# 최종 회고

## 1. 처음 세운 작업 계획과 실제 구현 순서는 같았는가?

## 2. 가장 어려웠던 부분은 무엇이었는가?

## 3. 처음에는 몰랐지만 이번 문제를 통해 알게 된 Python 문법 또는 개념은 무엇인가?

## 4. 반복문을 사용하면서 값이 어떻게 변하는지 설명할 수 있는가?

## 5. pandas의 sum(), mean(), max(), min()을 사용하지 않고 직접 구현해 보니 무엇을 이해하게 되었는가?

## 6. 아직 이해가 안 되는 부분은 무엇인가?

## 7. 같은 문제를 다시 푼다면 가장 먼저 무엇을 할 것인가?

## 8. AI를 사용했다면, AI에게 묻지 않고 스스로 해결할 수 있었던 질문은 무엇이었는가?

# 제출 정보

## GitHub Notebook URL
https://github.com/

## AI 사용 여부
사용함

## AI 채팅 공유 URL

### AI 채팅 1
https://